# Week 89, Day 1: Build

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w089d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')
with open('waymark_map.py', 'w', encoding='utf-8') as f:
    f.write('from waymark_domain import PathNetwork\n\nHEIGHTS = {"Mill": 40, "Weir": 55, "Church": 80, "Ridge": 310, "Farm": 120, "Wood": 150, "Lake": 60, "Island": 5}\nPATHS = [("Mill", "Weir", 1.5), ("Weir", "Church", 2.0), ("Mill", "Church", 4.0), ("Church", "Ridge", 3.0),\n         ("Weir", "Farm", 2.5), ("Farm", "Ridge", 2.0), ("Farm", "Wood", 1.5), ("Wood", "Lake", 2.0), ("Lake", "Mill", 3.5)]\n\n\ndef sample_network():\n    network = PathNetwork(HEIGHTS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n')
with open('waymark_geo.py', 'w', encoding='utf-8') as f:
    f.write('import math\nfrom waymark_domain import PathNetwork\nfrom waymark_map import HEIGHTS, PATHS\n\nPOSITIONS = {"Mill": (0.0, 0.0), "Weir": (1.4, 0.4), "Church": (2.8, 1.2), "Ridge": (4.4, 3.4), "Farm": (2.8, 2.4),\n             "Wood": (2.0, 3.5), "Lake": (0.6, 2.8), "Island": (-3.0, 5.0)}\n\n\nclass GeoNetwork(PathNetwork):\n    """A path network whose junctions also have positions, in kilometres east and north."""\n\n    def __init__(self, heights, positions):\n        super().__init__(heights)\n        self.positions = dict(positions)\n\n    def straight_km(self, a, b):\n        return math.dist(self.positions[a], self.positions[b])\n\n\ndef sample_geo():\n    network = GeoNetwork(HEIGHTS, POSITIONS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n\n\ndef grid_network(size):\n    """A size-by-size grid of junctions named \'r-c\', 1 km apart, with slightly longer paths to their neighbours."""\n    names = {(r, c): f"{r}-{c}" for r in range(size) for c in range(size)}\n    heights = {name: ((r * 13 + c * 7) % 9) * 20 for (r, c), name in names.items()}\n    positions = {name: (float(c), float(r)) for (r, c), name in names.items()}\n    network = GeoNetwork(heights, positions)\n    for (r, c), name in names.items():\n        if c + 1 < size:\n            network.add_path(name, names[(r, c + 1)], 1.0 + ((r * 7 + c * 3) % 5) / 10)\n        if r + 1 < size:\n            network.add_path(name, names[(r + 1, c)], 1.0 + ((r * 3 + c * 5) % 5) / 10)\n    return network\n')

**Predict before you run:** what will this print?



In [ ]:
import math
from waymark_geo import sample_geo

network = sample_geo()
print(network.positions["Ridge"], round(network.straight_km("Mill", "Ridge"), 2))
print(network.shortest("Mill", "Ridge").km)
for a, b in [("Mill", "Weir"), ("Farm", "Ridge")]:
    print(a, b, network.paths[a][b], round(network.straight_km(a, b), 2))

> **Key idea.** Measure before optimising, counting work (junctions finished) rather than timing. A* expands the junction with the smallest `g + h`; with an admissible heuristic (never an overestimate, like the straight-line distance) it still finds the shortest route while exploring less; an overestimate makes it wrong. Check the data supports the heuristic. Cache rarely changing data and invalidate with a version number that every change increases. Find nearest points with a grid index, searching outward ring by ring and stopping when no closer point is possible.

### Exercise: Counting the work

Write `dijkstra_work(network, start, end)`: Dijkstra's algorithm (visiting neighbours in name order, as the page's `waymark_domain` does) returning `(km, finished)`: the shortest distance rounded to 2 places and the number of junctions finished (popped and expanded for the first time) when the destination is finished, **including** the destination. Raise the domain's `NoRoute` if it can't be reached.

In [ ]:
import heapq
from waymark_domain import NoRoute
from waymark_geo import grid_network, sample_geo

def dijkstra_work(network, start, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_work():\n    \"how much did it explore?\"\n    assert dijkstra_work(sample_geo(), \"Mill\", \"Ridge\") == (6.0, 7)\n    grid = grid_network(30)\n    assert dijkstra_work(grid, \"0-0\", \"5-7\") == (13.3, 80)\n    assert dijkstra_work(grid, \"0-0\", \"29-29\") == (63.8, 900)\n    try:\n        dijkstra_work(sample_geo(), \"Mill\", \"Island\")\n    except NoRoute:\n        pass\n    else:\n        raise AssertionError(\"no route\")")

### Exercise: A*

Write `astar(network, start, end)` for a `GeoNetwork`: return `(route, finished)`, where `route` is a domain `Route` (stops, km rounded to 2 places, climb from `network.climb`) and `finished` counts junctions finished as in the previous exercise. Push `(g + h, name)` with `h = network.straight_km(name, end)`; expand neighbours in name order; raise `UnknownPlace` for unknown places and `NoRoute` as the domain does.

In [ ]:
import heapq
from waymark_domain import NoRoute, Route, UnknownPlace
from waymark_geo import grid_network, sample_geo

def astar(network, start, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_geo import grid_network, sample_geo\n\ndef test_astar():\n    \"same answers, less work\"\n    net = sample_geo()\n    route, work = astar(net, \"Mill\", \"Ridge\")\n    assert (route.stops, route.km, route.climb_m, work) == ([\"Mill\", \"Weir\", \"Farm\", \"Ridge\"], 6.0, 270, 4)\n    grid = grid_network(30)\n    route, work = astar(grid, \"0-0\", \"5-7\")\n    assert route.km == 13.3 and work == 48 and route.stops[0] == \"0-0\" and route.stops[-1] == \"5-7\"\n    for a, b in [(\"Lake\", \"Church\"), (\"Ridge\", \"Mill\"), (\"Wood\", \"Weir\")]:\n        assert astar(net, a, b)[0].km == net.shortest(a, b).km\n    for bad, kind in (((\"Mill\", \"Island\"), NoRoute), ((\"Moon\", \"Mill\"), UnknownPlace)):\n        try:\n            astar(net, *bad)\n        except kind:\n            pass\n        else:\n            raise AssertionError(bad)")

### Exercise: A cache that knows when the map changed

Write `MapCache(load, version)`: `load()` builds the network (expensive) and `version()` returns the current map version (cheap). `get()` returns the cached network, calling `load()` only the first time and whenever `version()` differs from the version it was loaded at; it counts `loads` and `hits`. `clear()` empties it, so the next `get()` loads.

In [ ]:
class MapCache:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cache():\n    \"load once per version\"\n    state = {\"version\": 1, \"loads\": 0}\n    def load():\n        state[\"loads\"] += 1\n        return {\"map\": state[\"version\"]}\n    cache = MapCache(load, lambda: state[\"version\"])\n    first = cache.get()\n    assert cache.get() is first and cache.get() is first and (cache.loads, cache.hits) == (1, 2)\n    state[\"version\"] = 2\n    second = cache.get()\n    assert second == {\"map\": 2} and second is not first and cache.loads == 2\n    cache.clear()\n    cache.get()\n    assert cache.loads == 3 and cache.hits == 2 and state[\"loads\"] == 3")

### Exercise: The nearest junction

Write `GridIndex(positions, cell=1.0)`: place each junction (`name -> (x, y)`) in the cell `(floor(x / cell), floor(y / cell))`. `nearest(x, y)` returns the name of the junction closest to the point (ties to the alphabetically first), searching ring by ring outward from the point's cell (ring 0 is the cell itself, ring `r` the cells exactly `r` cells away in either direction), and stopping after a ring once the best distance found is at most `r × cell` (no farther ring can hold anything closer). Count the cells looked at in `self.cells_checked` (reset on each call). An empty index returns `None`.

In [ ]:
import math

class GridIndex:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_geo import POSITIONS, grid_network\n\ndef test_grid():\n    \"rings, not everything\"\n    index = GridIndex(POSITIONS)\n    assert index.nearest(0.2, 0.1) == \"Mill\" and index.nearest(4.0, 3.0) == \"Ridge\" and index.nearest(-2.5, 4.4) == \"Island\"\n    brute = lambda x, y: min(POSITIONS, key=lambda n: (math.dist(POSITIONS[n], (x, y)), n))\n    import math\n    for x, y in [(1.0, 1.0), (2.4, 2.9), (3.6, 0.0), (-1.0, 2.0), (2.0, 3.0)]:\n        assert index.nearest(x, y) == brute(x, y), (x, y)\n    big = grid_network(30)\n    bindex = GridIndex(big.positions)\n    assert bindex.nearest(12.3, 7.6) == \"8-12\" and bindex.cells_checked <= 9\n    assert GridIndex({}).nearest(0, 0) is None")

### Exercise: Checking the heuristic's assumption

Write `too_short(network, tolerance=0.01)`: the paths (each once, as `(a, b)` with `a < b`, sorted) whose recorded length is shorter than the straight-line distance between their ends by more than `tolerance` km. Such paths make the straight-line heuristic overestimate, so A* may return a longer route.

In [ ]:
from waymark_geo import sample_geo

def too_short(network, tolerance=0.01):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_geo import sample_geo\n\ndef test_too_short():\n    \"the data must support the heuristic\"\n    net = sample_geo()\n    assert too_short(net) == []\n    net.paths[\"Mill\"][\"Lake\"] = net.paths[\"Lake\"][\"Mill\"] = 2.0\n    net.positions[\"Ridge\"] = (14.4, 3.4)\n    assert too_short(net) == [(\"Church\", \"Ridge\"), (\"Farm\", \"Ridge\"), (\"Lake\", \"Mill\")]")

### Bug hunt: A faster, longer route

Someone doubled the heuristic to make A* "more decisive". It is faster, but it no longer finds the shortest route: it prints `13.4`. Use the straight-line distance itself, so it prints `13.3`.

In [ ]:
import heapq
from waymark_geo import grid_network

network = grid_network(30)
start, end = "0-0", "5-7"
g, done, heap = {start: 0}, set(), [(0, start)]
while heap:
    _, node = heapq.heappop(heap)
    if node in done:
        continue
    done.add(node)
    if node == end:
        break
    for nxt, km in sorted(network.paths[node].items()):
        if g[node] + km < g.get(nxt, float("inf")):
            g[nxt] = g[node] + km
            heapq.heappush(heap, (g[nxt] + 2 * network.straight_km(nxt, end), nxt))
print(round(g[end], 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"never overestimate\"\n    expect_output(\"13.3\")")

### Bug hunt: Heap entries that can't be compared

The heap holds `(cost, route)` pairs. When two costs tie, Python compares the routes, which can't be ordered, and it crashes with `TypeError: '<' not supported`. Add a tie-breaker that can always be compared (a counter) between the cost and the route, so it prints `['A', 'B'] ['A', 'C']`.

In [ ]:
import heapq

class Route:
    def __init__(self, stops):
        self.stops = stops

heap = []
heapq.heappush(heap, (2.0, Route(["A", "B"])))
heapq.heappush(heap, (2.0, Route(["A", "C"])))
print(heapq.heappop(heap)[-1].stops, heapq.heappop(heap)[-1].stops)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a comparable tie-breaker\"\n    expect_output(\"['A', 'B'] ['A', 'C']\")")

### Puzzle: A* in one loop

Arrange A*'s main loop. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: A* with h = 0 everywhere behaves like Dijkstra's algorithm
q = ___   # the word for a heuristic that never overestimates
r = ___   # loads made by a version cache asked get() 10 times while the version changes once (after the 4th get)
s = ___   # the cell (as a tuple) of the point (2.7, -0.3) with 1 km cells
t = ___   # "count work" or "time it": the better first measure of an algorithm change

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q == \"admissible\" and r == 2, \"p, q and r\"\n    assert s == (2, -1) and t == \"count work\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: from where I'm standing

Write `Planner(load, version, cell=1.0)` combining today's pieces (given): a `MapCache` of the network, and a `GridIndex` built from its positions, rebuilt whenever the cache loads a new map. `plan_from(x, y, end)` snaps the point to the nearest junction, finds the route with `astar`, and returns a dict: `start` (the junction), `walk_to_start_km` (straight-line distance to it, 2 places), `stops`, `km` (the route's km plus the walk to the start, 1 place) and `minutes` (Naismith's rule on that total and the route's climb). Raise `NoRoute` as `astar` does, and `ValueError("no map loaded")` if the map has no junctions. Keep counters `searches` and, through the cache, `loads`.

In [ ]:
import heapq
import math
from waymark_domain import NoRoute, Route, UnknownPlace, naismith_minutes
from waymark_geo import GeoNetwork, sample_geo

def astar(network, start, end):
    g, prev, done = {start: 0}, {}, set()
    heap = [(network.straight_km(start, end), start)]
    while heap:
        _, node = heapq.heappop(heap)
        if node in done:
            continue
        done.add(node)
        if node == end:
            stops = [end]
            while stops[-1] != start:
                stops.append(prev[stops[-1]])
            return Route(stops[::-1], round(g[end], 2), network.climb(stops[::-1])), len(done)
        for nxt, km in sorted(network.paths[node].items()):
            if g[node] + km < g.get(nxt, float("inf")):
                g[nxt], prev[nxt] = g[node] + km, node
                heapq.heappush(heap, (g[nxt] + network.straight_km(nxt, end), nxt))
    raise NoRoute(f"no route from {start} to {end}")

class MapCache:
    def __init__(self, load, version):
        self._load, self._version, self._value, self._at = load, version, None, None
        self.loads = self.hits = 0
    def get(self):
        current = self._version()
        if self._value is not None and current == self._at:
            self.hits += 1
            return self._value
        self._value, self._at = self._load(), current
        self.loads += 1
        return self._value

class GridIndex:
    def __init__(self, positions, cell=1.0):
        self.positions, self.cell, self.cells = dict(positions), cell, {}
        for name, (x, y) in self.positions.items():
            self.cells.setdefault((math.floor(x / cell), math.floor(y / cell)), []).append(name)
    def nearest(self, x, y):
        if not self.positions:
            return None
        return min(self.positions, key=lambda n: (math.dist(self.positions[n], (x, y)), n))

class Planner:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_geo import sample_geo, grid_network, GeoNetwork\n\ndef test_plan():\n    \"snap, search, add the walk in\"\n    state = {\"version\": 1}\n    planner = Planner(sample_geo, lambda: state[\"version\"])\n    r = planner.plan_from(0.3, 0.4, \"Ridge\")\n    assert r == {\"start\": \"Mill\", \"walk_to_start_km\": 0.5, \"stops\": [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"], \"km\": 6.5, \"minutes\": 105}\n    planner.plan_from(4.0, 3.0, \"Mill\")\n    assert planner.searches == 2 and planner.cache.loads == 1\n    state[\"version\"] = 2\n    planner.plan_from(1.4, 0.5, \"Church\")\n    assert planner.cache.loads == 2\n\ndef test_new_map_and_errors():\n    \"a new map means a new index\"\n    maps = {1: sample_geo(), 2: grid_network(5)}\n    state = {\"version\": 1}\n    planner = Planner(lambda: maps[state[\"version\"]], lambda: state[\"version\"])\n    assert planner.plan_from(0.1, 0.1, \"Weir\")[\"start\"] == \"Mill\"\n    state[\"version\"] = 2\n    assert planner.plan_from(3.9, 0.2, \"0-0\")[\"start\"] == \"0-4\"\n    try:\n        Planner(sample_geo, lambda: 1).plan_from(-3, 5, \"Mill\")\n    except NoRoute:\n        pass\n    else:\n        raise AssertionError(\"island\")\n    try:\n        Planner(lambda: GeoNetwork({}, {}), lambda: 1).plan_from(0, 0, \"Mill\")\n    except ValueError as e:\n        assert str(e) == \"no map loaded\"")

## Warm-up

Every path is at least as long as the straight line between its ends: you can't walk from Mill to Ridge in less than 5.4 km, whatever the paths do. That simple fact is what makes today's faster search correct.

## Measure first

"Make it faster" starts with a number. Timing is noisy (other programs, caches, a browser tab), so for algorithms count the **work** instead: here, how many junctions the search **finishes** (pops from the heap and expands) before it reaches the destination. Dijkstra's algorithm finishes every junction closer to the start than the destination, in **every direction**: to walk a few kilometres east it explores just as far west. On a 30 by 30 grid, from one corner to a junction nearby, that's 80 junctions to find a route through 13.

Then measure **where** the time goes in the whole request, not just the algorithm: in Waymark, loading the map from the database for each request turns out to cost far more than the search (Week 65's profiling). Fix the biggest cost first.

## A*: searching towards the goal

**A\*** is Dijkstra's algorithm with a sense of direction. Instead of always expanding the junction with the smallest distance **so far** (`g`), it expands the one with the smallest `g + h`, where `h` is an **estimate** of the distance still to go. With `h` the straight-line distance to the destination, junctions heading away from it look worse and are left alone.

A* still finds the shortest route if the estimate is **admissible**: it never **over**estimates the true remaining distance. The straight line is admissible because no path can be shorter than the straight line between its ends. Overestimate (multiply `h` by 2, say) and A* gets faster but can return a longer route. The optimisation therefore depends on the **data**: a path recorded as shorter than the straight line (a typo in the map, or positions in the wrong units) makes `h` inadmissible. Check that assumption when importing.

## Not loading the map every time

The map changes rarely (when someone imports a new one) and is read on every request, which is the perfect case for a **cache** (Week 81): load it once, keep it in memory, and reuse it. The hard part, as always, is **invalidation**: after an import, the cached map is stale. Waymark keeps a `map_version` number in the database that every import increases; the cache remembers which version it holds and reloads only when the number has changed. Checking a number is far cheaper than loading thousands of rows.

## Where am I?

To plan "from here", Waymark needs the junction nearest to a point. Checking every junction works for 8 and is slow for 50,000. A **grid index** divides the map into square cells and records which junctions are in each; to find the nearest to a point, look in the point's own cell, then the ring of cells around it, then the next ring, stopping once no unexplored cell could hold anything closer than the best found so far. It's the same idea as a hash table (Week 44), keyed by position.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Measure** your slowest operation: count its work or profile it (Week 65), and write the number down before changing anything.
2. **Fix the biggest cost**, and measure again. Record both numbers in an ADR, with the assumption the optimisation depends on.
3. **A test for that assumption** (like `too_short`), so bad data can't silently break the fast path.
4. **Merge** and update the burndown.

## Before you go

Look at your forecast from the sprint review. Is this sprint's plan still realistic after today? Tomorrow is S4, round walks, done properly at last.